[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/poncelet/poncelet.ipynb)

# Poncelet's Porism

This example draws polygons inscribed in one quadric and circumscribed about another. From a point on an outer ellipsoid, a path follows a tangent of an inner quadric to where it meets the ellipsoid again, takes the other tangent from there, and goes on. Poncelet's porism states that if the path closes after some number of sides from one starting point, it closes after that number from every starting point, and if it misses from one, it misses from all.

Points are positions, antivectors, and the outer ellipsoid is a map from a point to its polar plane, `Plane <- Point`, zero on its points. The sides of the path are chords, joins of two points, and the inner quadric is held by the chords that touch it: it sends a chord to its pole, the meet of the polars of the chord's points, `Pole <- Chord`, and a chord touches it where it passes through its own pole. Everything about one quadric has a twin about the other: five points fix a quadric through them and five chords a quadric they touch; a chord from a point meets a quadric once more, and through a point pass two tangents. Knowing one of the two, the other follows without a square root, by the same formula, once along directions to the ellipsoid and once along directions to tangents.

In matrix notation the quadric and its envelope read as a symmetric matrix and its adjugate. In the language of elliptic curves, the pairs of a point on the outer quadric and a tangent of the inner one through it form a curve of genus one, a step of the path reads as adding a fixed point on it, and closing after $n$ sides as that point having order $n$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga
from examples.util.animation import save_figure, show_animation
from examples.math.poncelet import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
# Positions are points, intrinsic; polars are planes, read by which side a point is on, extrinsic.
Point = ga.gatype.antivector()                                                 # Antivector-
Plane = ga.gatype.vector()                                                     # Vector+
# A chord is the join of two points, a line oriented along itself; a pole is the meet of two planes.
Chord = (Point & Point).output_gatype                                          # Antibivector-
Pole = (Plane ^ Plane).output_gatype                                           # Bivector+
Quadric = ga.gatype((Plane, Point))                                            # Plane+ <- Point-
Envelope = ga.gatype((Pole, Chord))                                            # Pole+ <- Chord-
Motor = ga.gatype.rotor()                                                      # Rotor+

## 1. An ellipsoid, and a quadric touching a pentagon in it

The ellipsoid is a sum of dyads, each coordinate plane through the origin weighted by the inverse square of its semi-axis, less the plane at infinity, carried to its place by a motor. Two poles make a quadric held by its chords, their pair, zero on every chord through either; the pairs of opposite corners of four chords span every quadric touching those four, and a fifth chord picks one. The sides of a pentagon inscribed in the ellipsoid fix the first inner quadric; the sides of the same pentagon shrunk a little towards the centre fix the second.

In [ ]:
# The ellipsoid: its centre, its semi-axes, and how far its first axis is turned from x.
centre = np.array([0.4, -0.2])
semi = np.array([2.0, 1.2])
tilt = 0.5
# The corners of the pentagon, as angles around the ellipsoid, the next corner of each, and how far the
# second pentagon is shrunk towards the centre.
corner_angles = np.array([0.1, 0.8, 1.9, 3.0, 5.2])
following = np.array([1, 2, 3, 4, 0])
shrink = 0.97

In [ ]:
def ellipsoid(semi: np.ndarray, motor: Motor) -> Quadric:
    """The ellipsoid with the given semi-axes along x and y, carried from the origin by a motor."""
    axes = stack([mv.x, mv.y])                                                 # [2] Plane+
    at_origin = (axes * (axes & Point) / semi**2).sum(axis=-1) - mv.w * (mv.w & Point)   # [] Plane+ <- Point-
    return motor >> at_origin(motor << Point)                                  # [] Plane+ <- Point-


def envelope(sides: Chord) -> Envelope:
    """The envelope touching five chords on the last axis."""
    first, second, third, fourth, fifth = (sides[..., index] for index in range(5))
    # Two opposite corners of the quadrilateral of the first four chords, and two others.
    p, q = first ^ second, third ^ fourth                                      # [...] Pole+, Pole+
    r, s = first ^ third, second ^ fourth                                      # [...] Pole+, Pole+
    # Each pair of poles as one envelope, symmetric in the two.
    corners = (p * (q & Chord) + q * (p & Chord)) / 2                          # [...] Pole+ <- Chord-
    diagonals = (r * (s & Chord) + s * (r & Chord)) / 2                        # [...] Pole+ <- Chord-
    # The combination of the two pairs that vanishes on the fifth chord.
    return corners * (diagonals(fifth) & fifth) - diagonals * (corners(fifth) & fifth)   # [...] Pole+ <- Chord-


# The ellipsoid turned about the origin by the tilt and carried to its centre.
shift_to_centre = ((mv.xw * centre[0] + mv.yw * centre[1]) * 0.5).exp()       # [] Motor+
tilt_turn = (mv.xy * (-tilt / 2)).exp()                                        # [] Motor+
placement = shift_to_centre * tilt_turn                                        # [] Motor+


def on_ellipsoid(angles: np.ndarray, scale: float) -> Point:
    """Points at the given angles on the ellipsoid, drawn in towards its centre by the scale."""
    return placement >> mv.point(scale * semi * np.stack([np.cos(angles), np.sin(angles)], axis=-1))


outer = ellipsoid(semi, placement)                                             # [] Plane+ <- Point-
corners = stack([on_ellipsoid(corner_angles, 1.0), on_ellipsoid(corner_angles, shrink)])   # [inners, corners] Point-
inner = envelope(corners & corners[..., following])                            # [inners] Pole+ <- Chord-

In [ ]:
print("the ellipsoid:\n", outer.coefficient_table())
print("the quadric touching the pentagon, held by its chords:\n", inner[0].coefficient_table())

## 2. The other crossing and the other tangent

A chord through a vertex is the vertex joined with a direction, a point at infinity. Along a direction from a point of the ellipsoid, the ellipsoid's value is a quadratic with one root at the point, so the other crossing is rational in the quadric. Its twin on tangents: among the chords through a vertex, those that touch the inner quadric are the roots of a quadratic in the direction; knowing one, the incoming side, the other follows rationally from any second direction that is never a tangent, such as the direction from the ellipsoid's centre, which lies inside both inner quadrics. Only the first side from each start, with neither root known, takes a square root.

In [ ]:
# Both roots of a quadratic, one per sign.
signs = np.array([1, -1])


def tangents(inner: Envelope, points: Point) -> Point:
    """The directions of the two tangents of an envelope through each point, on the last axis: the
    roots of its quadratic along the directions, by a square root."""
    # The points at infinity along x and along y, and the chords through each point towards them.
    x_direction, y_direction = mv.x.dual(), mv.y.dual()                        # [] Point-, Point-
    x_chords, y_chords = points & x_direction, points & y_direction            # [...] Chord-, Chord-
    a = x_chords & inner(x_chords)                                             # [...] Scalar+
    b = x_chords & inner(y_chords)                                             # [...] Scalar+
    c = y_chords & inner(y_chords)                                             # [...] Scalar+
    # The discriminant is positive where two tangents pass, outside the inner quadric, and zero on it,
    # where they coincide.
    root = (b * b - a * c).square_root()                                       # [...] Scalar+
    # The roots of `a + 2 * b * t + c * t * t` in t, as the weights of the two directions.
    return x_direction * (root[..., None] * signs - b[..., None]) + y_direction * a[..., None]   # [..., 2] Point-


def path(outer: Quadric, inner: Envelope, start: Point, heading: Point, inside: Point) -> Iterator[Point]:
    """The vertices of the path on the outer quadric, from a start on it along the direction of a tangent
    of the inner quadric, without end: where that chord meets the outer quadric again, then the other
    tangent through the new vertex, turned against the direction from a point inside the inner quadric."""
    vertex = start.broadcast_to(heading.shape)                                 # [...] Point-
    while True:
        yield vertex
        # Along `vertex + heading * t` the outer quadric's pairing is
        # `2 * t * (outer(vertex) & heading) + t * t * (outer(heading) & heading)`, zero at the vertex and at the other root.
        crossing = vertex * (outer(heading) & heading) - heading * (2 * (outer(vertex) & heading))   # [...] Point-
        vertex = crossing.normalized()                                         # [...] Point-
        # The incoming side, and the chord from a point inside the inner quadric, which is never a tangent.
        spoke = vertex - inside                                                # [...] Point-
        side, spoke_line = vertex & heading, vertex & spoke                    # [...] Chord-, Chord-
        # Along `heading + spoke * t` the envelope's pairing of the chords through the vertex is
        # `2 * t * (side & inner(spoke_line)) + t * t * (spoke_line & inner(spoke_line))`, zero at the incoming side and at the other tangent.
        tangent = heading * (spoke_line & inner(spoke_line)) - spoke * (2 * (side & inner(spoke_line)))   # [...] Point-
        # A direction has no weight; its length is the norm of its complement.
        heading = tangent / tangent.dual().norm()                              # [...] Point-

## 3. Closing from every start, or from none

From starts all around the ellipsoid, the path of five sides about the first inner quadric always closes on its start, left; about the second, which touches no pentagon inscribed in the ellipsoid, it always misses, right. The start is orange and the fifth vertex ringed; each inner quadric is drawn where the two tangents through a point coincide.

In [ ]:
# The sides of each path; the starts around the ellipsoid, from the angle of the first; the frame shown as a
# still, and the duration of a frame in milliseconds.
n_sides = 5
n_frames = 120
first_start = 0.05
start_angles = np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False) + first_start
still = 10
frame_duration = 60

In [ ]:
starts = on_ellipsoid(start_angles, 1.0)                                       # [starts] Point-
first_heading = tangents(inner[:, None], starts[None, :])[..., 0]              # [inners, starts] Point-
# The ellipsoid's centre, inside both inner quadrics: no tangent of either passes through it.
inside = mv.point(centre)                                                      # [] Point-
paths = path(outer, inner[:, None], starts[None, :], first_heading, inside)
vertices = stack(list(islice(paths, n_sides + 1)))                             # [sides + 1, inners, starts] Point-
save_figure(render.draw(outer, inner, vertices[:, :, still]), "poncelet_pentagons")
show_animation(render.animate([(outer, inner, vertices[:, :, frame]) for frame in range(n_frames)]),
               "poncelet_turn", frame_duration)

In [ ]:
# How far the fifth vertex lands from the start, about each inner quadric, over all starts: the difference
# of two unit points is a direction, and its length is the norm of its complement.
gap = (vertices[n_sides] - vertices[0]).dual().norm()                          # [inners, starts] Scalar+
print("smallest gap about the second inner quadric:", gap[1].to_array().min())

In [ ]:
# checks
# Every vertex lies on the ellipsoid, and every side passes through its own pole of its inner quadric; the
# path about the first inner quadric closes after five sides from every start, the path about the second
# does not.
np.testing.assert_allclose((outer(vertices) & vertices).to_array(), 0.0, atol=1e-12)
sides = (vertices[:-1] & vertices[1:]).normalized()                            # [sides, inners, starts] Chord
touch = (sides & inner[:, None](sides)).to_array() / np.abs(inner.kernel).max(axis=(-1, -2))[:, None]
np.testing.assert_allclose(touch, 0.0, atol=1e-10)
np.testing.assert_allclose(gap[0].to_array(), 0.0, atol=1e-10)
assert (gap[1] > 1e-2).all()